# Tutorial: YOLO11n object detection inference

Prerequisites:
- Базове розуміння images як tensors.
- Розуміння різниці між training і inference.
- Встановлені залежності з `requirements-yolo.txt`.

Learning goals:
- Завантажити готову nano YOLO-модель.
- Запустити object detection на кількох зображеннях.
- Прочитати `class`, `confidence` і `bounding box` для detections.
- Зберегти annotated images.


## Outline

1. Перевіряємо файли й залежності.
2. Завантажуємо `yolo11n.pt`.
3. Запускаємо inference на demo-зображеннях.
4. Дивимося detections як таблицю.
5. Зберігаємо й показуємо annotated images.
6. Змінюємо confidence threshold.


In [1]:
from __future__ import annotations

from pathlib import Path

ROOT = Path.cwd()
MODEL_PATH = ROOT / "models" / "yolo11n.pt"
IMAGE_DIR = ROOT / "images"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

ROOT


WindowsPath('C:/Users/Student/Documents/Shtefan-Python/python-ai/заняття_12/12_yolo_inference')

## 1. Перевірка assets

Якщо файлів ваг моделі й demo-зображення немає, наступна клітинка запустить `prepare_yolo_assets.py`.


In [2]:
import subprocess
import sys

required_files = [MODEL_PATH, IMAGE_DIR / "bus.jpg", IMAGE_DIR / "zidane.jpg"]
missing = [path for path in required_files if not path.exists()]

if missing:
    subprocess.run([sys.executable, "prepare_yolo_assets.py"], check=True)

for path in required_files:
    print(f"{path.relative_to(ROOT)}: {path.exists()}")


models\yolo11n.pt: True
images\bus.jpg: True
images\zidane.jpg: True


## 2. Імпорт Ultralytics і завантаження моделі

`YOLO("models/yolo11n.pt")` завантажує готову модель object detection, вже навчену на COCO-класах. Це inference, не training.


In [3]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
print(type(model).__name__)


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Student\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO


## 3. Запуск inference

YOLO повертає список `Results`: по одному результату на кожне зображення. У кожному результаті є `boxes`, `names`, шлях до зображення і метод `plot()` для візуалізації.


In [4]:
image_paths = sorted(path for path in IMAGE_DIR.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print([path.name for path in image_paths])

results = model.predict(
    source=[str(path) for path in image_paths],
    conf=0.25,
    imgsz=640,
    device="cpu",
    verbose=False,
)

print(f"Results: {len(results)}")


['bus.jpg', 'zidane.jpg']
Results: 2


## 4. Detections як таблиця

Для кожного знайденого об'єкта збережемо:

- назву зображення;
- клас;
- confidence;
- координати bounding box `xmin, ymin, xmax, ymax`.


In [5]:
rows = []

for result in results:
    for box in result.boxes:
        class_id = int(box.cls.item())
        xmin, ymin, xmax, ymax = [float(value) for value in box.xyxy[0].tolist()]
        rows.append(
            {
                "image": Path(result.path).name,
                "class": result.names[class_id],
                "confidence": round(float(box.conf.item()), 3),
                "xmin": round(xmin, 1),
                "ymin": round(ymin, 1),
                "xmax": round(xmax, 1),
                "ymax": round(ymax, 1),
            }
        )


import pandas as pd

detections = pd.DataFrame(rows)
display(detections)



,image,class,confidence,xmin,ymin,xmax,ymax
0,image0.jpg,bus,0.939,11.9,228.4,799.2,735.2
1,image0.jpg,person,0.902,48.6,398.0,243.2,904.5
2,image0.jpg,person,0.849,670.6,392.6,810.0,879.6
3,image0.jpg,person,0.833,223.1,405.6,345.2,859.7
4,image0.jpg,person,0.399,0.0,550.2,66.0,871.8
5,image1.jpg,person,0.858,748.8,41.4,1149.4,710.8
6,image1.jpg,person,0.788,143.1,200.8,1134.8,712.4
7,image1.jpg,tie,0.484,361.0,438.3,524.1,715.9


## 5. Збереження annotated images

`result.plot()` малює bounding boxes, назви класів і confidence. Збережемо результат як звичайні `.jpg` файли.


In [6]:
from IPython.display import Image as DisplayImage, display
from PIL import Image

saved_images = []

for result in results:
    annotated_bgr = result.plot()
    annotated_rgb = annotated_bgr[..., ::-1]
    out_path = RESULTS_DIR / f"{Path(result.path).stem}_detected.jpg"
    Image.fromarray(annotated_rgb).save(out_path, quality=95)
    saved_images.append(out_path)
    print(out_path.relative_to(ROOT))


results\image0_detected.jpg
results\image1_detected.jpg


In [ ]:
for path in saved_images:
    display(DisplayImage(filename=str(path), width=640))


## 6. Вправа: змініть confidence threshold

Запустіть inference з `conf=0.5` або `conf=0.7` і порівняйте, які detections зникли. Вища межа confidence зазвичай зменшує кількість об'єктів, але не гарантує кращий результат для кожного зображення.


In [ ]:
exercise_results = model.predict(
    source=[str(path) for path in image_paths],
    conf=0.5,
    imgsz=640,
    device="cpu",
    verbose=False,
)

for result in exercise_results:
    print(Path(result.path).name, "detections:", len(result.boxes))


## Common pitfall

Не плутати `yolo11n.pt` inference з навчанням YOLO. Для training потрібен датасет зі спеціальною object detection розміткою: клас і bounding box для кожного об'єкта.

Optional extension:
- Додайте власне зображення в `images/` і повторіть inference.
- Спробуйте `--conf 0.1`, `--conf 0.5`, `--conf 0.7` у `run_yolo_demo.py`.
